### Changes to the fairness analysis

- Added `stratify=y` to the train/test split so the training and test sets retain similar proportions of good- and bad-credit cases.
- Set the positive outcome label to `1` (good credit), so TPR and PPV are calculated for that outcome.
- Added predictive parity, measured as positive predictive value (PPV), alongside TPR.
- To evaluate more groups, add the feature name and its value-to-group mapping to `group_definitions`.

In [18]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import os
  
# fetch dataset 
statlog_german_credit_data = fetch_ucirepo(id=144)

# data (as pandas dataframes) 
X = statlog_german_credit_data.data.features 
y = statlog_german_credit_data.data.targets 

# metadata 
print(statlog_german_credit_data.metadata) 
  
# variable information 
print(statlog_german_credit_data.variables)

{'uci_id': 144, 'name': 'Statlog (German Credit Data)', 'repository_url': 'https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data', 'data_url': 'https://archive.ics.uci.edu/static/public/144/data.csv', 'abstract': 'This dataset classifies people described by a set of attributes as good or bad credit risks. Comes in two formats (one all numeric). Also comes with a cost matrix', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 1000, 'num_features': 20, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Other', 'Marital Status', 'Age', 'Occupation'], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1994, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5NC77', 'creators': ['Hans Hofmann'], 'intro_paper': None, 'additional_info': {'summary': 'Two datasets are provided.  the original dataset, in the form provided by

In [19]:
column_names = {
    "Attribute1": "checking_status",
    "Attribute2": "duration_months",
    "Attribute3": "credit_history",
    "Attribute4": "purpose",
    "Attribute5": "credit_amount",
    "Attribute6": "savings",
    "Attribute7": "employment_since",
    "Attribute8": "installment_rate",
    "Attribute9": "personal_status_sex",
    "Attribute10": "other_debtors",
    "Attribute11": "residence_since",
    "Attribute12": "property",
    "Attribute13": "age_years",
    "Attribute14": "other_installment_plans",
    "Attribute15": "housing",
    "Attribute16": "existing_credits",
    "Attribute17": "job",
    "Attribute18": "dependents",
    "Attribute19": "telephone",
    "Attribute20": "foreign_worker",
}

X = X.rename(columns=column_names)
y = y.rename(columns={"class": "credit_risk"})

print(X.columns.tolist())
print(y.columns.tolist())

['checking_status', 'duration_months', 'credit_history', 'purpose', 'credit_amount', 'savings', 'employment_since', 'installment_rate', 'personal_status_sex', 'other_debtors', 'residence_since', 'property', 'age_years', 'other_installment_plans', 'housing', 'existing_credits', 'job', 'dependents', 'telephone', 'foreign_worker']
['credit_risk']


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Create a pipeline with preprocessing and the classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Train the model
pipeline.fit(X_train, y_train.values.ravel())
y_pred = pipeline.predict(X_test)

# Evaluate the model
accuracy = pipeline.score(X_test, y_test)
print(f"Model accuracy: {accuracy:.2f}")

Model accuracy: 0.74


/var/folders/zr/28ln2krd5q9d29k5y1pj_0sr0000gn/T/ipykernel_77090/2085463583.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object']).columns.tolist()


In [21]:
# Choose the outcome counted as positive: 1 = good credit, 2 = bad credit.
POSITIVE_LABEL = 1

group_definitions = {
    "Sex": (
        "personal_status_sex",
        {
            "A91": "Male",
            "A93": "Male",
            "A94": "Male",
            "A92": "Female",
            "A95": "Female",
        },
    ),
    "Foreign-worker status": (
        "foreign_worker",
        {
            "A201": "Foreign worker",
            "A202": "Not foreign worker",
        },
    ),
}

y_true = np.asarray(y_test).ravel()
y_pred_test = np.asarray(y_pred).ravel()

def calculate_tpr(y_true, y_pred, positive_label):
    actual_positive = y_true == positive_label
    if not actual_positive.any():
        return np.nan
    return np.mean(y_pred[actual_positive] == positive_label)

def calculate_ppv(y_true, y_pred, positive_label):
    predicted_positive = y_pred == positive_label
    if not predicted_positive.any():
        return np.nan  # PPV is undefined if no cases were predicted positive
    return np.mean(y_true[predicted_positive] == positive_label)

results = []

for attribute, (feature, mapping) in group_definitions.items():
    groups = X_test[feature].map(mapping)

    if groups.isna().any():
        unknown_values = X_test.loc[groups.isna(), feature].unique().tolist()
        raise ValueError(f"Unmapped values in {feature}: {unknown_values}")

    group_names = list(dict.fromkeys(mapping.values()))
    group_tprs = {}
    group_ppvs = {}

    for group in group_names:
        mask = groups.to_numpy() == group
        tpr = calculate_tpr(y_true[mask], y_pred_test[mask], POSITIVE_LABEL)
        ppv = calculate_ppv(y_true[mask], y_pred_test[mask], POSITIVE_LABEL)
        
        group_tprs[group] = tpr
        group_ppvs[group] = ppv

        results.append({
            "attribute": attribute,
            "group": group,
            "TPR": tpr,
            "PPV": ppv,
        })

    valid_tprs = [rate for rate in group_tprs.values() if np.isfinite(rate)]
    gap = max(valid_tprs) - min(valid_tprs) if len(valid_tprs) >= 2 else np.nan

    valid_ppvs = [rate for rate in group_ppvs.values() if np.isfinite(rate)]
    ppv_gap = max(valid_ppvs) - min(valid_ppvs) if len(valid_ppvs) >= 2 else np.nan
    
    print(f"{attribute} TPR gap: {gap:.4f}")
    print(f"{attribute} PPV gap: {ppv_gap:.4f}")

pd.DataFrame(results)

Sex TPR gap: 0.0209
Sex PPV gap: 0.0080
Foreign-worker status TPR gap: 0.1111
Foreign-worker status PPV gap: 0.2357


,attribute,group,TPR,PPV
0,Sex,Male,0.898990,0.773913
1,Sex,Female,0.878049,0.765957
2,Foreign-worker status,Foreign worker,0.888889,0.764331
3,Foreign-worker status,Not foreign worker,1.000000,1.000000
